In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)


In [ ]:
!pip install huggingface
!pip install -U datasets
#!pip install sentence-transformers #uncomment only if needed
!pip install faiss-cpu
!pip install einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
import pandas as pd
from datasets import load_dataset
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import scipy.spatial
from datasets import load_dataset



dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

hotel_reviews_Istanbul.csv: 0.00B [00:00, ?B/s]

hotel_reviews_San%20Francisco.csv: 0.00B [00:00, ?B/s]

hotel_reviews_london.csv: 0.00B [00:00, ?B/s]

hotel_reviews_nyc.csv: 0.00B [00:00, ?B/s]

hotel_reviews_paris.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5997 [00:00<?, ? examples/s]

In [ ]:
df=pd.DataFrame(dataset['train'])
df.head()
df_paris = df.loc[df.locality=='Paris']
df_paris.drop_duplicates()
reviews = df_paris['review_text'].tolist()

In [ ]:

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

def get_embeddings(data, model):
    """
    Generates embeddings for a list of text data.

    Args:
        data (list): A list of strings.
        model (SentenceTransformer): The pre-trained sentence transformer model.

    Returns:
        np.ndarray: The embeddings of the input data.
    """
    embeddings = model.encode(data, show_progress_bar=False).astype('float32')
    return embeddings

def search_faiss_index(query_embedding, faiss_index, k=5):
    """
    Performs a semantic search on a FAISS index using a query embedding.

    Args:
        query_embedding (np.ndarray): The embedding of the query string.
        faiss_index (faiss.IndexFlatIP): The FAISS index object.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        tuple: A tuple containing:
            - distances (np.ndarray): The distances of the retrieved neighbors.
            - indices (np.ndarray): The indices of the retrieved neighbors in the original data.
    """
    # Normalize the query embedding
    query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)
    distances, indices = faiss_index.search(query_embedding_normalized, k)
    return distances, indices

def create_faiss_index(embeddings):
    """
    Creates a FAISS index from a set of embeddings.

    Args:
        embeddings (np.ndarray): The embeddings to index.

    Returns:
        faiss.IndexFlatIP: The created FAISS index object.
    """
    # Normalize the embeddings
    embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Initialize the FAISS index with cosine similarity (Inner Product)
    index = faiss.IndexFlatIP(embeddings_normalized.shape[1])
    # Add the normalized embeddings to the index
    index.add(embeddings_normalized)
    return index

# Example usage (assuming 'reviews' and 'model' are defined from previous code):
reviews = df_paris['review_text'].tolist()
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
if torch.cuda.is_available():
    model = model.to('cuda')

review_embeddings = get_embeddings(reviews, model)
faiss_index = create_faiss_index(review_embeddings)



modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [ ]:
query = "Hotel with a view of the Eiffel tower."
query_embedding = get_embeddings([query], model)

k = 25
distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

print(f"Query: {query}")
print("Top hotel with similar reviews using FAISS:")
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    # Note: With normalized embeddings and IndexFlatIP, the distance is the inner product,
    # which is equal to the cosine similarity.
    print(f"Cosine Similarity: {distance:.4f}")
    print()

Query: Hotel with a view of the Eiffel tower.
Top hotel with similar reviews using FAISS:
1. Pullman Paris Eiffel Tower Hotel
Review: Excellent service. Stunning view of the Eiffel towere from our balcony, The room is gorgeous, comfortable and spacious. Definitely will be recommending this hotel to family and friends. If you’re looking for a hotel that has everything you need in Paris, luxury and view, this is the one.
Cosine Similarity: 0.8227

2. Pullman Paris Eiffel Tower Hotel
Review: If you stay at this hotel it is for the amazing views of the Eiffel Tower and for the pictures.  The photos and memories of being on the balcony looking at the Eiffel Tower in all its splendor cannot be denied.   When we arrived we were impressed with proximity of hotel to the Eiffel Tower. Definitely within walking distance, 7 minutes, we could even see our hotel when we climbed up the tower later that day . The view itself is spectacular!! We were upgraded to a suite top floor (9th floor) with a bal

In [ ]:
import json

def search_hotels_by_query(query, model, faiss_index, df, k=25):
    """
    Performs a semantic search for hotels based on a query using a FAISS index.

    Args:
        query (str): The query string.
        model (SentenceTransformer): The pre-trained sentence transformer model.
        faiss_index (faiss.IndexFlatIP): The FAISS index object.
        df (pd.DataFrame): The DataFrame containing hotel data.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        str: A JSON string containing the query and the top k search results.
    """
    query_embedding = get_embeddings([query], model)
    distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

    results = []
    for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
        results.append({
            "rank": i,
            "hotel_name": df.iloc[idx]['hotel_name'],
            "review_text": df.iloc[idx]['review_text'],
            "cosine_similarity": float(distance) # Convert float32 to standard float
        })

    output = {
        "query": query,
        "top_results": results }

    return results

# Example usage:
# Assuming 'model', 'faiss_index', and 'df_paris' are already defined from the preceding code.
# query = "Hotel with a view of the Eiffel tower."
# json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)



In [ ]:
query = "Hotel with a view of the Eiffel tower."
json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)

In [ ]:
json_output

[{'rank': 1,
  'hotel_name': 'Pullman Paris Eiffel Tower Hotel',
  'review_text': 'Excellent service. Stunning view of the Eiffel towere from our balcony, The room is gorgeous, comfortable and spacious. Definitely will be recommending this hotel to family and friends. If you’re looking for a hotel that has everything you need in Paris, luxury and view, this is the one.',
  'cosine_similarity': 0.8226630687713623},
 {'rank': 2,
  'hotel_name': 'Pullman Paris Eiffel Tower Hotel',
  'review_text': 'If you stay at this hotel it is for the amazing views of the Eiffel Tower and for the pictures.  The photos and memories of being on the balcony looking at the Eiffel Tower in all its splendor cannot be denied.   When we arrived we were impressed with proximity of hotel to the Eiffel Tower. Definitely within walking distance, 7 minutes, we could even see our hotel when we climbed up the tower later that day . The view itself is spectacular!! We were upgraded to a suite top floor (9th floor) wit

Setting up OpenRouter API for LLM Endpoint

In [ ]:
# Retrieve API key securely from Colab user data
from google.colab import userdata
# Retrieve the value of a saved environment variable named 'OPEN_ROUTER_API_KEY'.
OPEN_ROUTER_API_KEY = userdata.get('OPEN_ROUTER_API_KEY')



# Initialize the OpenAI-compatible client, but point it to OpenRouter's API instead of OpenAI's
# OpenRouter is a gateway to multiple LLMs like GPT, Claude, Mistral, and others, through one unified API
from openai import OpenAI
open_router_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",  # Set the API endpoint to OpenRouter (not OpenAI)
    api_key=OPEN_ROUTER_API_KEY               # Use your OpenRouter API key for authentication
)


Now, let's integrate everything by combining our Retrieval functiom with the Language Model to complete our RAG (Retrieval-Augmented Generation) pipeline.

In [ ]:
# Define a function that uses a language model to generate an answer based on a user's query
def generate_answer(query):
    # Build the prompt that will be sent to the LLM
    # The prompt includes:
    # - Instructions to clean and format the answer
    # - The user's original query
    # - The context retrieved from Qdrant (via semantic search)

    json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)
    prompt = f"""
    Based on the following query from a user, please generate a small answer
    focusing on the original query and the response given. The answer should be paragraphs.
    Remove the special characters and (/n), make the output clean and long.
    Please cite source for each part as [1][2].
    Just start with the answer, no need to give any salutations.

    ###########
    query:
    "{query}"

    ########

    context:
    "{json_output}"
    #####

    Return in Markdown format.
    """

    # Send the prompt to the LLM using streaming mode
    # This allows the response to be received in real-time, piece by piece
    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",  # Model to use (can be any OpenAI-compatible model, change the model here as needed)
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
        stream=True,  # Enable streaming so we get partial output as it generates
    )

    # Initialize a variable to hold the full response
    output_text = ""

    # Iterate through the streaming response chunks
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            output_text += content  # Append new content to the full output
            print(content, end="")  # Print each chunk live as it's received

    # Return both the final answer and the context used (for reference or display)
    return output_text,json_output


In [ ]:
query = "Hotels with a view of the Eiffel Tower"
response,sources = generate_answer(query)

Pullman Paris Eiffel Tower Hotel is highly recommended for its exceptional view of the Eiffel Tower, with several reviews emphasizing the stunning balcony vistas and the ability to see the tower from the room [1]. The hotel’s proximity to the Eiffel Tower allows guests to enjoy the iconic landmark within walking distance, and some reviewers noted that they could even spot their hotel while ascending the tower [2]. While the rooms are described as comfortable and spacious, a few mentions highlight that the view might be partially obstructed depending on the room’s location, though the overall experience remains positive [14].  

Hotel Marignan Champs-Elysees offers superior rooms with an excellent view of the Eiffel Tower, particularly the Eiffel Tower suite [3]. Located near the Champs-Elysees, it balances accessibility to major attractions with a quieter atmosphere compared to the bustling tourist areas [11]. Reviewers praised the hotel’s helpful staff and the romantic evening views o

Spreading to all Cities

In [ ]:
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")
df_all=pd.DataFrame(dataset['train'])
df_all.drop_duplicates()
reviews = df_all['review_text'].tolist()

In [ ]:
# Import necessary modules from the Qdrant client library
# Qdrant is a vector database that allows you to store and search high-dimensional vector embeddings efficiently
!pip install qdrant_client
from qdrant_client import QdrantClient, models

# Create a new Qdrant client instance using in-memory storage
# ":memory:" means the data will be stored temporarily in RAM (not saved to disk)
# Useful for testing or prototyping — everything is wiped when the program ends
client = QdrantClient(":memory:")

# Display the size (number of dimensions) of the text embeddings we generated earlier
# This is important because Qdrant needs to know the exact size of each vector to create a collection
text_embeddings_size = 768

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 10.5 MB/s eta 0:00:00


In [ ]:
df_all.shape

(5997, 14)

In [ ]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False}) with Transformer model: NomicBertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [ ]:
df_all.locality.value_counts()

,count
locality,
Istanbul,1200
San Francisco,1200
London,1200
Paris,1200
New York City,1197


In [ ]:
# Filter out rows with None review_text before converting to list
reviews_df = df_all.dropna(subset=['review_text'])
reviews = reviews_df['review_text'].tolist()



review_embeddings = get_embeddings(reviews, model)

In [ ]:


try:
    # Define the name of the collection we want to manage in Qdrant.
    # A collection in Qdrant is similar to a table in traditional databases —
    # it stores a group of vectors and their associated metadata.
    collection_name="hotel_reviews"

    # Check whether the collection already exists in Qdrant.
    # This avoids attempting to create a collection with a name that's already taken.
    if client.collection_exists(collection_name):
        # If the collection already exists, delete it to ensure we're starting fresh.
        # This is useful when we want to reset the state (e.g., during development or re-indexing).
        client.delete_collection(collection_name=collection_name)

        # Output a message confirming the collection was deleted successfully.
        print(f"Collection '{collection_name}' deleted successfully.")

    # Proceed to create a new collection regardless of whether it was previously deleted or not.
    # This ensures we always end up with a clean, newly-created collection.
    client.create_collection(
        collection_name=collection_name,  # The name of the new collection being created

        # Configure how vectors will be stored in this collection.
        # This includes the dimensionality (size) and the distance metric used for similarity.
        vectors_config=models.VectorParams(
            size=text_embeddings_size,       # The number of dimensions in each vector.
                                             # Must match the output size of your embedding model.
            distance=models.Distance.COSINE  # The distance function used for comparing vectors.
                                             # COSINE is commonly used for text embeddings as it measures angular similarity.
        ),
    )

    # Print a confirmation that the collection was created successfully.
    print(f"Collection '{collection_name}' created successfully.")

except Exception as e:
    # If any error occurs during the process (e.g., connection issues, invalid parameters),
    # it will be caught here and the error message will be printed.
    print(f"An error occurred while setting up the collection: {e}")


Collection 'hotel_reviews' deleted successfully.
Collection 'hotel_reviews' created successfully.


In [ ]:

# Prepare the data for uploading to Qdrant
# We need to create "points", where each point contains:
# 1. An `id` (unique identifier for the point/review)
# 2. A `vector` (the embedding of the review text)
# 3. `payload` (metadata associated with the vector, like hotel name, city, etc.)


# Example usage (assuming 'reviews' and 'model' are defined from previous code):

points_to_upload = [
    models.PointStruct(
        id=i,  # Use the index as the unique ID
        vector=review_embeddings[i].tolist(),  # Convert numpy array to list for Qdrant
        payload={
            "hotel_name": reviews_df.iloc[i]['hotel_name'],
            "review_text": reviews_df.iloc[i]['review_text'],
            "locality": reviews_df.iloc[i]['locality'], # Include locality for filtering
        },
    )
    for i in range(len(review_embeddings))
]

# Upload the created points to the Qdrant collection in batches
client.upsert(
    collection_name="hotel_reviews",
    wait=True,  # Wait until the operation is completed
    points=points_to_upload,
)

# Function to search Qdrant with an optional city filter
def search_qdrant_with_filter(query, model, client, city=None, k=10):
    """
    Performs a semantic search on the Qdrant collection with an optional city filter.

    Args:
        query (str): The query string.
        model (SentenceTransformer): The pre-trained sentence transformer model.
        client (QdrantClient): The Qdrant client instance.
        city (str, optional): The name of the city to filter by. Defaults to None.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        list: A list of search results from Qdrant.
    """
    # Generate the embedding for the query
    query_embedding = model.encode(query, show_progress_bar=False).tolist()

    # Define the filter
    # If a city is specified, create a filter that matches the 'locality' field
    # Otherwise, the filter is None (no filtering)
    query_filter = None
    if city:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="locality",
                    match=models.MatchValue(value=city)
                )
            ]
        )

    # Perform the search


    text_hits = client.query_points(
    collection_name="hotel_reviews",  # The name of the collection where vectors were stored
    query=query_embedding,
    query_filter=query_filter,   # The query vector — what we want to find similar results to
    limit=k,                             # Limit the number of results to 3 most relevant chunks
    with_payload=True, # Include the payload (metadata) in the results
).points                                 # Extract only the list of matching points (each with vector + payload)



    return text_hits





In [ ]:
# Search for hotels with a view of the Eiffel Tower specifically in Paris
query = "Amazing hotel close to everythings"
city_filter = "Istanbul"
text_hits = search_qdrant_with_filter(query, model, client, city=city_filter, k=10)

In [ ]:
text_hits

[ScoredPoint(id=160, version=0, score=0.7180620899345458, payload={'hotel_name': 'White House Hotel Istanbul', 'review_text': '  One of the best\xa0experiences\xa0I’ve had at a hotel before. The hotel is amazing, highly beautiful and\xa0unique.\xa0I picked this hotel as boutique choice and it did not disappoint. Price was just slightly higher than other hotels but it’s very very nice and feels luxurious. It’s also has a great location, walking distance to the main attractions as well as plenty restaurants in the neighbour. Once you walk into this hotel you feel the warmth immediately, they also have a helpful concierge that made some dinner reservations for us! The staff are always welcoming, kind and always available. The rooms are nice,decent sized and you get a mini bar stocked daily. the breakfast is AMAZING, it was so good we just kept going back every single morning. I think the breakfast buffet is great for', 'locality': 'Istanbul'}, vector=None, shard_key=None, order_value=None

In [ ]:
# Search for hotels with a generic query and city name
query = "Amazing hotel close to everythings"
city_filter = "Istanbul"
qdrant_results_city = search_qdrant_with_filter(query, model, client, city=city_filter, k=10)

print(f"Query: {query} in {city_filter}")
print("Top hotel reviews using Qdrant with city filter:")
for i, result in enumerate(qdrant_results_city, 1):
    print(f"{i}. Hotel Name: {result.payload['hotel_name']}")
    print(f"   Review: {result.payload['review_text']}")
    print(f"   Score (Cosine Similarity): {result.score:.4f}")
    print(f"   Locality: {result.payload['locality']}")
    print()

# Search for hotels with a view of the Eiffel Tower without a specific city filter (searches across all data loaded into the collection)
query = "Hotel close to Hagia Sofia."
qdrant_results_all = search_qdrant_with_filter(query, model, client, city=None, k=10)

print(f"\nQuery: {query} (without city filter)")
print("Top hotel reviews using Qdrant (no filter):")
for i, result in enumerate(qdrant_results_all, 1):
    print(f"{i}. Hotel Name: {result.payload['hotel_name']}")
    print(f"   Review: {result.payload['review_text']}")
    print(f"   Score (Cosine Similarity): {result.score:.4f}")
    print(f"   Locality: {result.payload['locality']}")
    print()




Query: Amazing hotel close to everythings in Istanbul
Top hotel reviews using Qdrant with city filter:
1. Hotel Name: White House Hotel Istanbul
   Review:   One of the best experiences I’ve had at a hotel before. The hotel is amazing, highly beautiful and unique. I picked this hotel as boutique choice and it did not disappoint. Price was just slightly higher than other hotels but it’s very very nice and feels luxurious. It’s also has a great location, walking distance to the main attractions as well as plenty restaurants in the neighbour. Once you walk into this hotel you feel the warmth immediately, they also have a helpful concierge that made some dinner reservations for us! The staff are always welcoming, kind and always available. The rooms are nice,decent sized and you get a mini bar stocked daily. the breakfast is AMAZING, it was so good we just kept going back every single morning. I think the breakfast buffet is great for
   Score (Cosine Similarity): 0.7181
   Locality: Istan

In [ ]:
# Modify the generate_answer function to use Qdrant search
def generate_answer_qdrant(query, city=None):
    """
    Generates an answer using a language model based on a user's query
    and context retrieved from Qdrant, with an optional city filter.

    Args:
        query (str): The user's query string.
        city (str, optional): The name of the city to filter Qdrant search results by. Defaults to None.

    Returns:
        tuple: A tuple containing:
            - output_text (str): The generated answer from the LLM.
            - sources (list): The search results from Qdrant used as context.
    """
    # Use the new search_qdrant_with_filter function
    qdrant_results = search_qdrant_with_filter(query, model, client, city=city, k=25)

    # Format the Qdrant results into a string that the LLM can understand
    # You might want to adjust this formatting based on the LLM you use
    context_string = ""
    for i, result in enumerate(qdrant_results):
        context_string += f"Source {i+1}:\n"
        context_string += f"Hotel: {result.payload.get('hotel_name', 'N/A')}\n"
        context_string += f"Review: {result.payload.get('review_text', 'N/A')}\n"
        context_string += f"Locality: {result.payload.get('locality', 'N/A')}\n"
        context_string += f"Similarity Score: {result.score:.4f}\n\n"


    # Build the prompt for the LLM
    prompt = f"""
    Based on the following query from a user and the provided context from hotel reviews,
    please generate a concise answer summarizing the relevant information.
    Focus on addressing the user's query using details found in the reviews.
    Cite the sources using numerical references like [1], [2], etc., corresponding to the "Source #" in the context.
    Format the output as a few paragraphs.
    Remove any special characters like (/n) and ensure the output is clean.
    Begin directly with the answer.

    ###########
    query:
    "{query}"

    ########

    context:
    "{context_string}"
    #####

    Return in Markdown format.
    """

    # Send the prompt to the LLM using streaming mode
    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",  # Model to use (change as needed)
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
        stream=True,  # Enable streaming
    )

    # Initialize a variable to hold the full response
    output_text = ""

    # Iterate through the streaming response chunks and print them
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            output_text += content
            print(content, end="")

    # Return the generated text and the Qdrant search results
    return output_text, qdrant_results

In [ ]:
# Query for hotels with generic example
query= "Amazing hotel close to everything"
city_filter = "Istanbul"
response_paris, sources_paris = generate_answer_qdrant(query, city=city_filter)

print("\n--- Generated Answer (City Filter) ---")
# The response was already printed chunk by chunk inside the function.
# You can print the final output_text here again if needed:
# print(response_paris)

print("\n--- Sources Used (City Filter) ---")
# Print details about the sources (Qdrant results) that were used
for i, result in enumerate(sources_paris):
    print(f"Source {i+1}: Hotel: {result.payload.get('hotel_name', 'N/A')}, Locality: {result.payload.get('locality', 'N/A')}, Score: {result.score:.4f}")

print("\n" + "="*50 + "\n") # Separator


Several hotels in Istanbul are highlighted for their exceptional location and proximity to major attractions and amenities. The White House Hotel Istanbul is praised for its beautiful, unique design and convenient location, with a walking distance to main attractions and numerous restaurants nearby [1]. Skalion Hotel & Spa is noted for being within walking distance of iconic sites like the Blue Mosque and Hagia Sophia, and it offers attached markets, pharmacies, and restaurants, eliminating the need for taxis [2]. Hotel Amira Istanbul is frequently mentioned as a top choice, with its perfect location near attractions and helpful staff who even organized transport and provided itinerary guidance [4]. Tomtom Suites also stands out for its fantastic location and friendly staff, who offered excellent recommendations for dining and made breakfast available in the hotel’s restaurant [5]. 

Other notable options include the Mula Hotel, which is just a 10-minute walk from landmarks such as Aya